In [75]:
import io, warnings
import pandas as pd
import matplotlib.pyplot as plt
import gcamreader
from lxml import etree
warnings.filterwarnings("ignore")

SPEIZER_DIR = "../scenarios/speizer2023/10028915/steel_decarb_runs_output"
SCENARIO_MAP = {
    "Reference_1":   "Speizer_Reference_1",
    "1p5C_12":       "Speizer_1p5C",
    "1p5C_13":       "Speizer_1p5C_noCCS",
    "1p5C_delay_14": "Speizer_1p5C_delay",
}
LABELS = {
    "Reference_1":   "Reference",
    "1p5C_12":       "1.5C all-adv",
    "1p5C_13":       "1.5C no-CCS",
    "1p5C_delay_14": "1.5C delay",
}
MTC_TO_MTCO2 = 44/12
TC_TO_TCO2   = 44/12


In [76]:
def load_speizer(fname, id_cols):
    chunks, header, buf = [], None, []
    for line in open(f"{SPEIZER_DIR}/{fname}").readlines()[1:]:
        if line.startswith("scenario,"):
            if header and buf:
                try: chunks.append(pd.read_csv(io.StringIO(header+"".join(buf))))
                except: pass
            header, buf = line, []
        elif header and line.strip(): buf.append(line)
    if header and buf:
        try: chunks.append(pd.read_csv(io.StringIO(header+"".join(buf))))
        except: pass
    df = pd.concat(chunks, ignore_index=True)
    df["scen"] = df["scenario"].str.split(",").str[0]
    year_cols = [c for c in df.columns if str(c).isdigit()]
    return (
        df[df["scen"].isin(SCENARIO_MAP)][id_cols+["scen"]+year_cols]
        .melt(id_vars=id_cols+["scen"], var_name="year", value_name="value")
        .assign(year=lambda d: d["year"].astype(int))
        .dropna(subset=["value"])
    )

# Speizer: global steel CO2
sp_co2 = (
    load_speizer("queryoutall_CO2_emissions_sector_nobio_v2.csv", ["region","sector"])
    .query('sector=="iron and steel"')
    .groupby(["scen","year"], as_index=False)["value"].sum()
    .assign(MtCO2=lambda d: d["value"]*MTC_TO_MTCO2)
)

# Speizer: global steel production
sp_prod = (
    load_speizer("queryoutall_ironsteel_production_tech.csv", ["region","technology"])
    .groupby(["scen","year"], as_index=False)["value"].sum()
)

# Speizer: carbon price
sp_price = (
    load_speizer("queryoutall_CO2_prices.csv", ["region","market"])
    .query('market=="globalCO2" and region=="Global"')
    .assign(price=lambda d: d["value"]/TC_TO_TCO2)
)

# Our DB
our_co2 = our_prod = our_price = pd.DataFrame()
conn, completed = None, []
try:
    conn = gcamreader.querymi.LocalDBConn("../output", "speizer_basexdb")
    avail = conn.listScenariosInDB()["name"].tolist()
    completed = [s for s in SCENARIO_MAP.values() if any(s in a for a in avail)]
    rev = {v:k for k,v in SCENARIO_MAP.items()}
    tree = etree.parse("../output/queries/Main_queries.xml")
    def qry(title, tag="supplyDemandQuery"):
        for t in [tag,"emissionsQueryBuilder","supplyDemandQuery"]:
            for el in tree.getroot().iter(t):
                if el.get("title")==title:
                    return conn.runQuery(gcamreader.querymi.Query(el), scenarios=completed)
        return pd.DataFrame()
    raw = qry("CO2 emissions by sector (excluding resource production)", "emissionsQueryBuilder")
    if not raw.empty:
        our_co2 = (raw[raw["sector"]=="iron and steel"]
            .groupby(["scenario","Year"],as_index=False)["value"].sum()
            .assign(scen=lambda d: d["scenario"].str.split(",").str[0].map(rev),
                    year=lambda d: d["Year"], MtCO2=lambda d: d["value"]*MTC_TO_MTCO2)
            .dropna(subset=["scen"]))
    raw = qry("iron and steel production by tech")
    if not raw.empty:
        our_prod = (raw.groupby(["scenario","Year"],as_index=False)["value"].sum()
            .assign(scen=lambda d: d["scenario"].str.split(",").str[0].map(rev),
                    year=lambda d: d["Year"])
            .dropna(subset=["scen"]))
    pol = [SCENARIO_MAP[k] for k in ["1p5C_12","1p5C_13","1p5C_delay_14"] if SCENARIO_MAP[k] in completed]
    if pol:
        raw = conn.runQuery(gcamreader.querymi.Query(
            next(el for el in tree.getroot().iter("supplyDemandQuery") if el.get("title")=="CO2 prices")
        ), scenarios=pol)
        if not raw.empty:
            our_price = (raw.assign(
                scen=lambda d: d["scenario"].str.split(",").str[0].map(rev),
                year=lambda d: d["Year"],
                price=lambda d: d["value"]/TC_TO_TCO2)
                .dropna(subset=["scen"]))
    print("Completed:", completed)
except Exception as e:
    print("DB:", e)


Database scenarios: Speizer_Reference_1
Completed: ['Speizer_Reference_1']
